# Problem Set 6: Randomized Experiments vs. Non-Experimental Comparison Groups — The LaLonde Data

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due Wednesday 11/4 (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

If you did not have a randomized experiment, how close could you get to the truth using non-experimental data and statistical adjustment alone?

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up

Throughout this course we have used a randomized controlled trial (the Oregon Medicaid lottery) as the gold-standard benchmark. This problem set asks the opposite question: **if you did *not* have a randomized experiment, how close could you get to the truth using non-experimental data and statistical adjustment alone?**

This is one of the oldest and most influential exercises in causal inference, from LaLonde (1986, *AER*) and the follow-up by Dehejia and Wahba (1999, *JASA*).

The **National Supported Work (NSW) Demonstration** was a real randomized job-training program in the 1970s for severely disadvantaged workers. Applicants were *randomly assigned* to subsidized, supportive work experience or not; the outcome is real earnings in 1978. Because assignment was random, the simple difference in mean 1978 earnings is a credible estimate of the program's effect — our benchmark "truth."

LaLonde's famous contribution: throw away the experimental control group, compare treated workers instead to ordinary people from the **Current Population Survey (CPS)** who never applied, and standard methods badly fail to reproduce the benchmark — sometimes even getting the *sign* wrong.

| Variable | Description |
|---|---|
| `treat` | =1 if NSW treatment group, 0 if control/comparison |
| `age`, `educ` | Age in years; years of education |
| `black`, `hisp`, `marr`, `nodegree` | Demographic indicators |
| `re74`, `re75` | Real earnings 1974, 1975 (pre-treatment, $) |
| `re78` | Real earnings 1978 (**outcome**, $) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

In [ ]:
nsw = pd.read_csv("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/nsw_dw.csv")      # experimental sample: treated + randomized controls
cps = pd.read_csv("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/cps_controls.csv")  # non-experimental CPS comparison group

print("NSW:", nsw.shape, "| CPS:", cps.shape)
nsw.head()

## Part 1: The experimental benchmark

**Question 1.** How many treated and control observations are there? Compute mean `re78` for each group and the difference — your **experimental benchmark**.

In [ ]:
print(nsw.treat.value_counts().rename({1: "treated", 0: "control"}), "\n")
means = nsw.groupby("treat").re78.mean()
benchmark = means[1] - means[0]
print(f"Mean re78, treated: ${means[1]:,.0f}")
print(f"Mean re78, control: ${means[0]:,.0f}")
print(f"Difference (experimental benchmark): ${benchmark:,.0f}")

**Question 2.** Run this as a regression instead of a difference in means. Confirm the coefficient matches, and say whether it is statistically significant.

In [ ]:
exp1 = smf.ols("re78 ~ treat", nsw).fit(cov_type="HC1")
print(exp1.summary().tables[1])

**Answer:** *[your answer here]*

**Question 3.** Compare treated and control groups on the pre-treatment characteristics. **Do the two groups look similar? Why would we expect this, given how the NSW sample was constructed?**

In [ ]:
covars = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]

bal = nsw.groupby("treat")[covars].mean().T
bal.columns = ["control", "treated"]
bal["difference"] = bal.treated - bal.control
bal.round(2)

**Answer:** *[your answer here]*

## Part 2: What if we had no experimental control group?

Now set the experimental controls aside. Imagine you observed only the 185 treated NSW workers, but you have the large CPS survey and decide to use ordinary Americans as your comparison group.

**Question 4.** Append the treated NSW observations to the CPS sample and compare means. **How different do these groups look compared to Question 3? Which group is more economically advantaged?**

In [ ]:
treated = nsw[nsw.treat == 1]
obs = pd.concat([treated, cps], ignore_index=True)   # 185 treated + 15,992 CPS

comp = pd.DataFrame({"NSW treated": treated[covars].mean(),
                     "CPS comparison": cps[covars].mean()})
comp["difference"] = comp["NSW treated"] - comp["CPS comparison"]
comp.round(2)

**Answer:** *[your answer here]*

**Question 5.** Run the naive regression on this combined sample. **What do you find, and how does it compare to the experimental benchmark?**

You should find something startling. Describe it, and explain why comparing treated workers to ordinary CPS respondents is so misleading — think about who applies to a program like NSW in the first place, and what their earnings look like just beforehand (the **"Ashenfelter dip"**).

In [ ]:
obs1 = smf.ols("re78 ~ treat", obs).fit(cov_type="HC1")
print(obs1.summary().tables[1])
print(f"\nNaive estimate: ${obs1.params['treat']:,.0f}   vs   benchmark: ${benchmark:,.0f}")

**Answer:** *[your answer here]*

## Part 3: Can statistical adjustment fix the problem?

**Question 6.** Re-run the naive regression controlling for all the covariates. Does it move closer to the benchmark? **By how much of the original gap does it close?**

In [ ]:
controls = " + ".join(covars)
obs2 = smf.ols(f"re78 ~ treat + {controls}", obs).fit(cov_type="HC1")

gap_before = obs1.params["treat"] - benchmark
gap_after = obs2.params["treat"] - benchmark
print(f"Naive:            ${obs1.params['treat']:>9,.0f}   (gap vs benchmark: ${gap_before:>9,.0f})")
print(f"OLS with controls:${obs2.params['treat']:>9,.0f}   (gap vs benchmark: ${gap_after:>9,.0f})")
print(f"Share of gap closed: {1 - abs(gap_after)/abs(gap_before):.1%}")

**Answer:** *[your answer here]*

**Question 7. Propensity scores.** Estimate a logit of `treat` on the covariates (plus a few squared terms for flexibility), predict the propensity score, and plot its distribution for each group. **What does this figure tell you about how comparable the two groups are? What fraction of the CPS sample has a score anywhere near that of most treated workers?**

In [ ]:
pscore_model = smf.logit(
    f"treat ~ {controls} + I(age**2) + I(re74**2) + I(re75**2)", obs).fit(disp=False)
obs["pscore"] = pscore_model.predict(obs)

plt.figure(figsize=(9, 5))
bins = np.linspace(0, 1, 60)
plt.hist(obs.loc[obs.treat == 0, "pscore"], bins=bins, alpha=0.6, label="CPS comparison", density=True)
plt.hist(obs.loc[obs.treat == 1, "pscore"], bins=bins, alpha=0.6, label="NSW treated", density=True)
plt.yscale("log")
plt.xlabel("Estimated propensity score"); plt.ylabel("Density (log scale)")
plt.legend(); plt.title("Propensity score distributions")
plt.show()

share = (obs[obs.treat == 0].pscore > 0.1).mean()
print(f"Share of CPS observations with pscore > 0.1: {share:.2%}")

**Answer:** *[your answer here]*

**Question 8. Common support.** Keep only observations whose propensity score lies within the range spanned by the treated group — this drops CPS respondents who look like nobody who was ever treated. **How many CPS observations survive? How does the estimate compare to the benchmark now?**

In [ ]:
lo, hi = obs.loc[obs.treat == 1, "pscore"].min(), obs.loc[obs.treat == 1, "pscore"].max()
trimmed = obs[(obs.pscore >= lo) & (obs.pscore <= hi)].copy()
print(f"Common support: [{lo:.4f}, {hi:.4f}]")
print(f"CPS observations remaining: {(trimmed.treat == 0).sum():,} of {(obs.treat == 0).sum():,}")

obs3 = smf.ols(f"re78 ~ treat + {controls}", trimmed).fit(cov_type="HC1")
print(f"\nTrimmed OLS estimate: ${obs3.params['treat']:,.0f}   vs benchmark ${benchmark:,.0f}")

**Answer:** *[your answer here]*

**Question 9. Inverse propensity weighting (IPW).** Weight treated observations by 1 and comparison observations by $\hat p/(1-\hat p)$, then re-run on the full untrimmed sample.

In [ ]:
obs["ipw"] = np.where(obs.treat == 1, 1.0, obs.pscore / (1 - obs.pscore))

obs4 = smf.wls(f"re78 ~ treat + {controls}", obs, weights=obs.ipw).fit(cov_type="HC1")
print(f"IPW estimate: ${obs4.params['treat']:,.0f}   vs benchmark ${benchmark:,.0f}")

**Answer:** *[your answer here]*

**Question 10. Checking covariate balance.** For each covariate compute the standardized percentage difference in means,

$$100 \times \frac{\bar X_{\text{treated}} - \bar X_{\text{control}}}{\sqrt{(s^2_{\text{treated}} + s^2_{\text{control}})/2}}$$

first with the unweighted CPS sample and then using IPW weights, and plot both as a **Love plot**. **Does IPW improve balance? Is balance fully restored for every covariate?**

In [ ]:
def std_diff(t, c, weights=None):
    mt, vt = t.mean(), t.var()
    if weights is None:
        mc, vc = c.mean(), c.var()
    else:
        w = weights / weights.sum()
        mc = (c * w).sum()
        vc = (w * (c - mc) ** 2).sum()
    return 100 * (mt - mc) / np.sqrt((vt + vc) / 2)

t_df = obs[obs.treat == 1]
c_df = obs[obs.treat == 0]
love = pd.DataFrame({
    "unweighted": [std_diff(t_df[v], c_df[v]) for v in covars],
    "IPW weighted": [std_diff(t_df[v], c_df[v], c_df.ipw) for v in covars],
}, index=covars)

plt.figure(figsize=(8, 5))
y = np.arange(len(covars))
plt.scatter(love.unweighted, y, label="Unweighted", s=60)
plt.scatter(love["IPW weighted"], y, label="IPW weighted", s=60, marker="s")
plt.axvline(0, color="black", lw=1)
for x in (-10, 10):
    plt.axvline(x, color="gray", ls="--", lw=1)
plt.yticks(y, covars); plt.xlabel("Standardized % difference in means")
plt.title("Love plot: covariate balance before and after IPW")
plt.legend(); plt.show()

love.round(1)

**Answer:** *[your answer here]*

## Part 4: Putting it all together

**Question 11.** Compare all your estimates side by side. **Which methods get closest to the experimental benchmark? Does any non-experimental method fully recover it?**

In [ ]:
results = pd.DataFrame({
    "estimate": [benchmark, obs1.params["treat"], obs2.params["treat"],
                 obs3.params["treat"], obs4.params["treat"]],
}, index=["(1) Experimental benchmark", "(2) Naive treated vs CPS", "(3) OLS with controls",
          "(4) OLS on common support", "(5) IPW"])
results["gap vs benchmark"] = results.estimate - benchmark

plt.figure(figsize=(9, 4.5))
colors = ["C2"] + ["C0"] * 4
plt.barh(results.index[::-1], results.estimate[::-1], color=colors[::-1])
plt.axvline(benchmark, color="C2", ls="--", label="Experimental benchmark")
plt.xlabel("Estimated effect on 1978 earnings ($)")
plt.legend(); plt.tight_layout(); plt.show()

results.round(0)

**Answer:** *[your answer here]*

**Question 12. The big picture.** In your own words: what is the general lesson of this exercise for anyone trying to evaluate a program or policy *without* a randomized experiment? Why might even a fairly careful non-experimental analysis still leave you meaningfully off from the truth?

**Answer:** *[your answer here]*

## Suggested additional questions (optional)

1. **Which controls matter most?** Redo Questions 5–6 controlling only for `age` and `educ`, dropping `re74` and `re75`. How much of the bias is removed without pre-treatment earnings? What does that tell you about which variables matter most when constructing a comparison group?
2. **Sensitivity to the trimming rule.** Try a stricter common-support rule (middle 90% of the treated score distribution) and a looser one. How sensitive is your answer?
3. **Nearest-neighbour matching.** As an alternative to IPW, match each treated worker to their nearest comparison neighbour by propensity score and compare estimates.
4. **External validity.** NSW targeted a very specific, severely disadvantaged population in the mid-1970s labour market. What reservations would you have about generalizing to a different population or era?
5. **Connecting back to the course.** Across the quarter we use four very different empirical strategies: a randomized experiment (Oregon Medicaid, PS5), the propensity-score and matching methods of today's problem set, and — still to come — staggered difference-in-differences (social media and mental health, PS7) and regression discontinuity (drinking age, PS8). Rank the two designs you have seen so far from most to least credible, and say what has to be true for each to identify a causal effect. Is today's exercise less a research design in its own right, and more a set of tools for *approximating* the experiment?

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS06"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")